In [0]:
# ===========================================
# Notebook Name:
# 00_inventory_existing_tables
#
# Purpose:
# Inventory all pokemon.{bronze,silver,gold,ops}
# tables and columns, and classify each against
# the v2 target shape defined in
# docs/migration_plan.
#
# Output:
# pokemon.ops.migration_inventory
#
# v2_action values:
# KEEP, MIGRATE, REBUILD, ARCHIVE,
# DROP_AFTER_VALIDATION
# ===========================================

In [0]:
from pyspark.sql import functions as F

CATALOG_NAME = "pokemon"

SCHEMAS = ["bronze", "silver", "gold", "ops"]

MIGRATION_INVENTORY_TABLE = (
    "pokemon.ops.migration_inventory"
)

print("Catalog:", CATALOG_NAME)
print("Schemas:", SCHEMAS)
print("Output :", MIGRATION_INVENTORY_TABLE)

In [0]:
# v2_action for tables that already exist today.
# Anything that exists but is not listed here is
# treated as an unexpected/legacy table.
V2_ACTION_BY_TABLE = {
    "bronze.tournament_list_raw": "KEEP",
    "bronze.event_result_raw": "KEEP",
    "bronze.deck_raw": "KEEP",
    "bronze.scrape_log": "KEEP",
    "bronze.tournament_list_scrape_log": "ARCHIVE",
    "silver.tournaments": "MIGRATE",
    "silver.tournament_results": "KEEP",
    "silver.decks": "KEEP",
    "silver.deck_cards": "KEEP",
    "gold.card_usage": "KEEP",
    "gold.deck_registry": "KEEP",
    "gold.deck_pokemon_features": "KEEP",
    "gold.deck_similarity": "KEEP",
    "gold.deck_archetypes": "KEEP",
    "ops.result_fetch_targets": "KEEP",
    "ops.deck_fetch_targets": "KEEP",
    "ops.pipeline_run_log": "KEEP",
    "ops.pipeline_state": "KEEP",
    "ops.migration_inventory": "KEEP",
}

DEFAULT_ACTION_FOR_UNKNOWN_TABLE = (
    "ARCHIVE"
)

# v2-standard tables that are expected to exist
# per docs/migration_plan section 12. Anything
# in this set but missing from the catalog is
# reported as REBUILD (needs to be created).
V2_TARGET_TABLES = {
    "bronze.tournament_list_raw",
    "bronze.event_result_raw",
    "bronze.deck_raw",
    "bronze.scrape_log",
    "silver.tournaments",
    "silver.tournament_results",
    "silver.decks",
    "silver.deck_cards",
    "gold.card_usage",
    "gold.deck_registry",
    "gold.deck_pokemon_features",
    "gold.deck_similarity",
    "gold.deck_archetypes",
    "ops.result_fetch_targets",
    "ops.deck_fetch_targets",
    "ops.pipeline_run_log",
    "ops.pipeline_state",
    "ops.migration_inventory",
}

print(
    "v2 target table count:",
    len(V2_TARGET_TABLES),
)

In [0]:
schema_list_sql = ", ".join(
    f"'{schema}'" for schema in SCHEMAS
)

existing_tables_df = spark.sql(f"""
    SELECT
        table_schema,
        table_name,
        created,
        last_altered
    FROM {CATALOG_NAME}.information_schema.tables
    WHERE table_schema IN ({schema_list_sql})
    ORDER BY table_schema, table_name
""")

existing_tables = (
    existing_tables_df.collect()
)

print(
    "Existing tables found:",
    len(existing_tables),
)

display(existing_tables_df)

In [0]:
existing_columns_df = spark.sql(f"""
    SELECT
        table_schema,
        table_name,
        column_name,
        ordinal_position,
        data_type,
        is_nullable
    FROM {CATALOG_NAME}.information_schema.columns
    WHERE table_schema IN ({schema_list_sql})
    ORDER BY
        table_schema,
        table_name,
        ordinal_position
""")

columns_by_table = {}

for row in existing_columns_df.collect():
    key = (
        row["table_schema"],
        row["table_name"],
    )
    columns_by_table.setdefault(
        key, []
    ).append(row)

print(
    "Tables with column metadata:",
    len(columns_by_table),
)

In [0]:
from datetime import datetime, timezone

inventory_rows = []

existing_table_keys = set()

for table_row in existing_tables:
    schema_name = table_row["table_schema"]
    table_name = table_row["table_name"]
    full_name = (
        f"{CATALOG_NAME}.{schema_name}."
        f"{table_name}"
    )
    spec_key = f"{schema_name}.{table_name}"
    existing_table_keys.add(spec_key)

    v2_action = V2_ACTION_BY_TABLE.get(
        spec_key,
        DEFAULT_ACTION_FOR_UNKNOWN_TABLE,
    )

    row_count = spark.table(
        full_name
    ).count()

    columns = columns_by_table.get(
        (schema_name, table_name), []
    )

    for column in columns:
        inventory_rows.append((
            CATALOG_NAME,
            schema_name,
            table_name,
            column["column_name"],
            column["data_type"],
            column["is_nullable"] == "YES",
            row_count,
            table_row["created"],
            table_row["last_altered"],
            v2_action,
        ))

print(
    "Inventory rows from existing tables:",
    len(inventory_rows),
)

In [0]:
missing_tables = sorted(
    V2_TARGET_TABLES - existing_table_keys
)

print("Missing v2 tables:", missing_tables)

for spec_key in missing_tables:
    schema_name, table_name = (
        spec_key.split(".", 1)
    )
    inventory_rows.append((
        CATALOG_NAME,
        schema_name,
        table_name,
        None,
        None,
        None,
        0,
        None,
        None,
        "REBUILD",
    ))

print(
    "Total inventory rows:",
    len(inventory_rows),
)

In [0]:
from pyspark.sql.types import (
    BooleanType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

INVENTORY_SCHEMA = StructType([
    StructField("catalog", StringType(), False),
    StructField("schema", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("column_name", StringType(), True),
    StructField("data_type", StringType(), True),
    StructField("nullable", BooleanType(), True),
    StructField("row_count", LongType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("last_modified", TimestampType(), True),
    StructField("v2_action", StringType(), False),
])

migration_inventory_df = spark.createDataFrame(
    inventory_rows,
    schema=INVENTORY_SCHEMA,
).withColumn(
    "inventoried_at",
    F.current_timestamp(),
)

In [0]:
display(
    migration_inventory_df
    .groupBy("schema", "table_name", "v2_action")
    .agg(F.max("row_count").alias("row_count"))
    .orderBy("schema", "table_name")
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {MIGRATION_INVENTORY_TABLE}
(
    catalog STRING NOT NULL,
    schema STRING NOT NULL,
    table_name STRING NOT NULL,
    column_name STRING,
    data_type STRING,
    nullable BOOLEAN,
    row_count BIGINT,
    created_at TIMESTAMP,
    last_modified TIMESTAMP,
    v2_action STRING NOT NULL,
    inventoried_at TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Snapshot of current-state table inventory vs. v2 migration_plan target shape. Overwritten on every run.'
""")

spark.sql(
    f"TRUNCATE TABLE {MIGRATION_INVENTORY_TABLE}"
)

(
    migration_inventory_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        MIGRATION_INVENTORY_TABLE
    )
)

print(
    "migration_inventory rebuilt with",
    migration_inventory_df.count(),
    "rows",
)

In [0]:
display(
    spark.table(MIGRATION_INVENTORY_TABLE)
    .select("schema", "table_name", "v2_action")
    .distinct()
    .orderBy("schema", "table_name")
)